In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import gc
import pickle
import keras
from keras import layers
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
import matplotlib.pyplot as plt

In [ ]:
def evaluate_disentanglement(latents_pred, true_factors):
    n_latents = latents_pred.shape[1]
    n_factors = true_factors.shape[1]

    latents_centered = latents_pred - latents_pred.mean(axis=0)
    u, s, vh = np.linalg.svd(latents_centered, full_matrices=False)
    s_norm = s / s.sum()
    effective_rank = np.exp(-np.sum(s_norm * np.log(s_norm + 1e-10)))

    mi_matrix = np.zeros((n_factors, n_latents))
    for f in range(n_factors):
        mi_matrix[f, :] = mutual_info_regression(latents_pred, true_factors[:, f])
    mig_scores = []
    for f in range(n_factors):
        sorted_mi = np.sort(mi_matrix[f, :])[::-1]
        factor_entropy = mutual_info_regression(true_factors[:, f].reshape(-1, 1), true_factors[:, f])[0]
        gap = (sorted_mi[0] - sorted_mi[1]) / (factor_entropy + 1e-8)
        mig_scores.append(gap)
    mig = np.mean(mig_scores)

    importance_matrix = np.zeros((n_factors, n_latents))
    r2_scores = []
    for f in range(n_factors):
        rf = RandomForestRegressor(n_estimators=50, random_state=0)
        rf.fit(latents_pred, true_factors[:, f])
        importance_matrix[f, :] = rf.feature_importances_
        r2_scores.append(rf.score(latents_pred, true_factors[:, f]))

    latent_importance_norm = importance_matrix / (importance_matrix.sum(axis=0, keepdims=True) + 1e-8)
    entropy_per_latent = -np.sum(latent_importance_norm * np.log(latent_importance_norm + 1e-8), axis=0)
    disentanglement_per_latent = 1 - entropy_per_latent / np.log(n_factors)
    weight_per_latent = importance_matrix.sum(axis=0) / importance_matrix.sum()
    D = np.sum(disentanglement_per_latent * weight_per_latent)

    factor_importance_norm = importance_matrix / (importance_matrix.sum(axis=1, keepdims=True) + 1e-8)
    entropy_per_factor = -np.sum(factor_importance_norm * np.log(factor_importance_norm + 1e-8), axis=1)
    completeness_per_factor = 1 - entropy_per_factor / np.log(n_latents)
    C = np.mean(completeness_per_factor)

    I = np.mean(r2_scores)
    return mig, D, C, I, effective_rank

In [ ]:
import numpy as np
import gc

FILE_PATH = 'E:/research_vae2/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz'

with np.load(FILE_PATH, allow_pickle=True) as data:
    imgs_full = data['imgs']
    latents_classes = data['latents_classes']

max_pixel = imgs_full.max()

N_TOTAL = 20000
total_size = len(imgs_full)
idx = np.linspace(0, total_size - 1, N_TOTAL, dtype=int)

np.random.RandomState(42).shuffle(idx)

x_subset = imgs_full[idx]
factors_subset = latents_classes[idx]

del imgs_full, latents_classes, data
gc.collect()


if max_pixel > 1.0:
    x_subset = x_subset.astype("float32") / 255.0
else:
    x_subset = x_subset.astype("float32")

x_subset = np.expand_dims(x_subset, axis=-1)


x_train = x_subset[:18000]
x_test = x_subset[18000:]
factors_test = factors_subset[18000:]

del x_subset, factors_subset
gc.collect()



In [ ]:
def build_small_no_res():
    inputs = keras.Input(shape=(64, 64, 1))
    x = layers.Flatten()(inputs)
    h1 = layers.Dense(64, activation="relu")(x)
    h2 = layers.Dense(32, activation="relu")(h1)
    z = layers.Dense(10)(h2)
    d1 = layers.Dense(32, activation="relu")(z)
    d2 = layers.Dense(64, activation="relu")(d1)
    d_out = layers.Dense(64*64*1, activation="sigmoid")(d2)
    outputs = layers.Reshape((64, 64, 1))(d_out)
    return keras.Model(inputs, outputs), keras.Model(inputs, z)

def build_big_no_res():
    inputs = keras.Input(shape=(64, 64, 1))
    x = layers.Flatten()(inputs)
    h1 = layers.Dense(1024, activation="relu")(x)
    h2 = layers.Dense(512, activation="relu")(h1)
    h3 = layers.Dense(256, activation="relu")(h2)
    h4 = layers.Dense(128, activation="relu")(h3)
    h5 = layers.Dense(64, activation="relu")(h4)
    z = layers.Dense(10)(h5)
    d1 = layers.Dense(64, activation="relu")(z)
    d2 = layers.Dense(128, activation="relu")(d1)
    d3 = layers.Dense(256, activation="relu")(d2)
    d4 = layers.Dense(512, activation="relu")(d3)
    d5 = layers.Dense(1024, activation="relu")(d4)
    d_out = layers.Dense(64*64*1, activation="sigmoid")(d5)
    outputs = layers.Reshape((64, 64, 1))(d_out)
    return keras.Model(inputs, outputs), keras.Model(inputs, z)

def build_small_res():
    inputs = keras.Input(shape=(64, 64, 1))
    x = layers.Flatten()(inputs)
    h1 = layers.Dense(64)(x);  s1 = layers.Dense(64)(x)
    h1 = layers.ReLU()(layers.Add()([h1, s1]))
    h2 = layers.Dense(32)(h1); s2 = layers.Dense(32)(h1)
    h2 = layers.ReLU()(layers.Add()([h2, s2]))
    z = layers.Dense(10)(h2)
    d1 = layers.Dense(32)(z);  sd1 = layers.Dense(32)(z)
    d1 = layers.ReLU()(layers.Add()([d1, sd1]))
    d2 = layers.Dense(64)(d1); sd2 = layers.Dense(64)(d1)
    d2 = layers.ReLU()(layers.Add()([d2, sd2]))
    d_out = layers.Dense(64*64*1, activation="sigmoid")(d2)
    outputs = layers.Reshape((64, 64, 1))(d_out)
    return keras.Model(inputs, outputs), keras.Model(inputs, z)

def build_big_res():
    inputs = keras.Input(shape=(64, 64, 1))
    x = layers.Flatten()(inputs)
    h1 = layers.Dense(1024)(x);  s1 = layers.Dense(1024)(x)
    h1 = layers.ReLU()(layers.Add()([h1, s1]))
    h2 = layers.Dense(512)(h1);  s2 = layers.Dense(512)(h1)
    h2 = layers.ReLU()(layers.Add()([h2, s2]))
    h3 = layers.Dense(256)(h2);  s3 = layers.Dense(256)(h2)
    h3 = layers.ReLU()(layers.Add()([h3, s3]))
    h4 = layers.Dense(128)(h3);  s4 = layers.Dense(128)(h3)
    h4 = layers.ReLU()(layers.Add()([h4, s4]))
    h5 = layers.Dense(64)(h4);   s5 = layers.Dense(64)(h4)
    h5 = layers.ReLU()(layers.Add()([h5, s5]))
    z = layers.Dense(10)(h5)
    d1 = layers.Dense(64)(z);   sd1 = layers.Dense(64)(z)
    d1 = layers.ReLU()(layers.Add()([d1, sd1]))
    d2 = layers.Dense(128)(d1); sd2 = layers.Dense(128)(d1)
    d2 = layers.ReLU()(layers.Add()([d2, sd2]))
    d3 = layers.Dense(256)(d2); sd3 = layers.Dense(256)(d2)
    d3 = layers.ReLU()(layers.Add()([d3, sd3]))
    d4 = layers.Dense(512)(d3); sd4 = layers.Dense(512)(d3)
    d4 = layers.ReLU()(layers.Add()([d4, sd4]))
    d5 = layers.Dense(1024)(d4); sd5 = layers.Dense(1024)(d4)
    d5 = layers.ReLU()(layers.Add()([d5, sd5]))
    d_out = layers.Dense(64*64*1, activation="sigmoid")(d5)
    outputs = layers.Reshape((64, 64, 1))(d_out)
    return keras.Model(inputs, outputs), keras.Model(inputs, z)

In [ ]:
import gc
import pickle
import numpy as np
import tensorflow as tf
import keras

DATASET_NAME = "dSprites"

def train_model_and_log(model_name, builder_fn):

    keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(0)
    np.random.seed(0)

    autoencoder, encoder = builder_fn()
    autoencoder.compile(optimizer="adam", loss="mse")

    history_dict = {'rank': [], 'mig': [], 'dci_d': [], 'dci_c': []}

    def on_epoch_end(epoch, logs):
        latents = encoder.predict(x_test[:500], verbose=0) 
        centered = latents - latents.mean(axis=0)
        u, s, vh = np.linalg.svd(centered, full_matrices=False)
        s_norm = s / s.sum()
        rank = np.exp(-np.sum(s_norm * np.log(s_norm + 1e-10)))
        history_dict['rank'].append(rank)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            mig, D, C, I, _ = evaluate_disentanglement(latents, factors_test[:500])
            history_dict['mig'].append(mig)
            history_dict['dci_d'].append(D)
            history_dict['dci_c'].append(C)

    callback = tf.keras.callbacks.LambdaCallback(on_epoch_end=on_epoch_end)

    history = autoencoder.fit(
        x_train, x_train,
        validation_data=(x_test, x_test),
        epochs=50,
        batch_size=32,
        verbose=0,
        callbacks=[callback]
    )

    history_dict['loss'] = history.history['loss']
    history_dict['val_loss'] = history.history['val_loss']

    with open(f'{DATASET_NAME}_{model_name}_curve_new.pkl', 'wb') as f:
        pickle.dump(history_dict, f)

    del autoencoder, encoder
    gc.collect()
    keras.backend.clear_session()

In [ ]:
train_model_and_log("Small_No_Res", build_small_no_res)

In [ ]:
train_model_and_log("Small_Res", build_small_res)

In [ ]:
train_model_and_log("Big_No_Res", build_big_no_res)

In [ ]:
train_model_and_log("Big_Res", build_big_res)

In [ ]:
model_names = ["Small_No_Res", "Small_Res", "Big_No_Res", "Big_Res"]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for i, model_name in enumerate(model_names):
    with open(f'{DATASET_NAME}_{model_name}_curve_new.pkl', 'rb') as f:
        h = pickle.load(f)

    axes[0, i].plot(h['loss'], label='Train', linewidth=2)
    axes[0, i].plot(h['val_loss'], label='Val', linewidth=2)
    axes[0, i].set_title(f'{model_name} - Loss', fontsize=12)
    axes[0, i].set_xlabel('Epoch'); axes[0, i].set_ylabel('MSE')
    axes[0, i].legend(); axes[0, i].grid(alpha=0.3)

    axes[1, i].plot(h['rank'], color='red', linewidth=2)
    axes[1, i].axhline(y=10, color='gray', linestyle='--', label='Max Capacity')
    axes[1, i].set_title(f'{model_name} - Effective Rank', fontsize=12)
    axes[1, i].set_xlabel('Epoch'); axes[1, i].set_ylabel('Rank')
    axes[1, i].set_ylim(0, 10.5); axes[1, i].legend(); axes[1, i].grid(alpha=0.3)

plt.suptitle(f'Training Dynamics on {DATASET_NAME}', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{DATASET_NAME}_training_curves_new.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
disent_epochs = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45,50]

for model_name in model_names:
    with open(f'{DATASET_NAME}_{model_name}_curve_new.pkl', 'rb') as f:
        h = pickle.load(f)
    n = len(h['mig'])
    epochs_used = disent_epochs[:n]

    axes[0].plot(epochs_used, h['mig'], marker='o', label=model_name, linewidth=2)
    axes[1].plot(epochs_used, h['dci_d'], marker='o', label=model_name, linewidth=2)
    axes[2].plot(epochs_used, h['dci_c'], marker='o', label=model_name, linewidth=2)

axes[0].set_title('MIG over Training'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('DCI-D over Training'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].set_title('DCI-C over Training'); axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f'Disentanglement Trajectories on {DATASET_NAME}', fontsize=14)
plt.tight_layout()
plt.savefig(f'{DATASET_NAME}_disentanglement_trajectories_new.png', dpi=150, bbox_inches='tight')
plt.show()